# 01 — Data Exploration

**Goal:** Get familiar with the play-by-play data. Understand what a game looks like as a stochastic process before we model it.

**What we're looking at:**
- Score margin over time for individual games
- Distribution of margin changes per minute (this is the σ we'll estimate)
- Whether the data looks clean (monotonic scores, no missing quarters, etc.)

**Prerequisites:** Run `python -m infra.db.init_db` and load at least one season with `/load-season`.

In [ ]:
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

import sys
sys.path.insert(0, str(Path('..').resolve()))  # so we can import infra/notebooks

from infra.db import queries

DB_PATH = '../data/quant_lab.db'

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('muted')

print('Setup complete.')

## 1. What games do we have?

Check how many games are in the database and their date range.

In [ ]:
games = queries.get_games(DB_PATH, league='NBA')
print(f"Total NBA games: {len(games)}")
print(f"Date range: {games['game_date'].min()} → {games['game_date'].max()}")
print(f"Seasons: {sorted(games['season'].unique())}")
games.head()

## 2. Score margin over time — a single game

Plot home − away score margin as a function of elapsed minutes for one game.
If our Brownian motion model is reasonable, this should look like a random walk with drift.

In [ ]:
# Pick a sample game
sample_game_id = games.iloc[0]['game_id']
pbp = queries.get_play_by_play(DB_PATH, sample_game_id)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(pbp['elapsed_minutes'], pbp['score_margin'], linewidth=1.5, color='steelblue')
ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)

# Mark quarter breaks
for q_end in [12, 24, 36, 48]:
    ax.axvline(q_end, color='lightgray', linestyle=':', linewidth=1)

ax.set_xlabel('Elapsed minutes')
ax.set_ylabel('Score margin (home − away)')
ax.set_title(f'Score margin over time — Game {sample_game_id}')
plt.tight_layout()
plt.show()

print(f"Final margin: {pbp['score_margin'].iloc[-1]} ({'home win' if pbp['score_margin'].iloc[-1] > 0 else 'away win'})")

## 3. Distribution of margin changes

The key question: **what does dX(t) look like?**

If score differential is Brownian motion with constant σ, the 1-minute margin changes should be approximately normally distributed with mean ≈ 0 and std ≈ σ.

In [ ]:
# Load PBP for all games to get the full distribution
pbp_all = queries.get_play_by_play_for_sigma(DB_PATH, league='NBA')
print(f"Total PBP rows: {len(pbp_all):,}")
print(f"Games covered: {pbp_all['game_id'].nunique():,}")
pbp_all.head()

In [ ]:
# Compute 1-minute margin changes within each game
pbp_all = pbp_all.sort_values(['game_id', 'elapsed_minutes'])

# Bucket elapsed_minutes into 1-minute intervals
pbp_all['minute_bucket'] = pbp_all['elapsed_minutes'].apply(lambda x: int(x))

# For each game, take the last margin reading per minute bucket
minute_margins = (
    pbp_all
    .groupby(['game_id', 'minute_bucket'])['score_margin']
    .last()
    .reset_index()
)

# Compute first differences within each game
minute_margins['margin_change'] = minute_margins.groupby('game_id')['score_margin'].diff()
changes = minute_margins['margin_change'].dropna()

print(f"1-min margin changes:")
print(f"  Mean:  {changes.mean():.3f}  (should be near 0 for a balanced dataset)")
print(f"  Std:   {changes.std():.3f}  ← this is our empirical σ")
print(f"  N:     {len(changes):,}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Histogram with normal overlay
from scipy.stats import norm
ax = axes[0]
ax.hist(changes, bins=40, density=True, alpha=0.7, color='steelblue', label='Observed')
x = np.linspace(changes.min(), changes.max(), 200)
ax.plot(x, norm.pdf(x, changes.mean(), changes.std()), 'r-', linewidth=2, label='Normal fit')
ax.set_xlabel('1-min margin change (points)')
ax.set_ylabel('Density')
ax.set_title('Distribution of 1-min margin changes')
ax.legend()

# Q-Q plot to check normality
from scipy.stats import probplot
ax2 = axes[1]
probplot(changes, dist='norm', plot=ax2)
ax2.set_title('Q-Q plot (normal)')

plt.tight_layout()
plt.show()

## 4. Summary stats

Collect the numbers we'll need for model calibration in the next notebook.

In [ ]:
sigma_empirical = changes.std()

print("=" * 40)
print("SUMMARY FOR MODEL CALIBRATION")
print("=" * 40)
print(f"Empirical σ (1-min):  {sigma_empirical:.4f} points/√min")
print(f"Games in dataset:     {pbp_all['game_id'].nunique():,}")
print()
print("Take these numbers to 02_parameter_estimation.ipynb for formal estimation.")